In [1]:
import json
import pandas as pd

records = []

with open("RAG data/train_Data1.json", "r", encoding="utf-8") as f:

    for line in f:
        line = line.strip()

        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Shape: (701, 4)
Columns: ['content', 'annotation', 'extras', 'metadata']


,content,annotation,extras,metadata
0,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta...",None,"{'first_done_at': 1527844872000, 'last_updated..."
1,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80...",None,"{'first_done_at': 1527845028000, 'last_updated..."
2,Anvitha Rao\nAutomation developer\n\n- Email m...,"[{'label': ['Links'], 'points': [{'start': 288...",None,"{'first_done_at': 1527744637000, 'last_updated..."
3,arjun ks\nSenior Program coordinator - oracle ...,"[{'label': ['Skills'], 'points': [{'start': 50...",None,"{'first_done_at': 1527834843000, 'last_updated..."
4,"Arun Elumalai\nQA Tester\n\nChennai, Tamil Nad...","[{'label': ['Skills'], 'points': [{'start': 19...",None,"{'first_done_at': 1527847268000, 'last_updated..."


In [2]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.13.0+cpu
CUDA available: False


In [3]:
print("Total rows:", len(df))

duplicate_count = df["content"].duplicated().sum()
unique_count = df["content"].nunique()

print("Unique resumes:", unique_count)
print("Duplicate rows:", duplicate_count)

Total rows: 701
Unique resumes: 536
Duplicate rows: 165


In [4]:
df_clean = df.drop_duplicates(
    subset=["content"]
).reset_index(drop=True)

print("Original:", len(df))
print("After removing duplicates:", len(df_clean))

Original: 701
After removing duplicates: 536


In [5]:
# Check for missing resume text
print("Missing resumes:", df_clean["content"].isna().sum())

# Check for empty resumes
print("Empty resumes:", (df_clean["content"].str.strip() == "").sum())

# Resume length
df_clean["text_length"] = df_clean["content"].str.len()

print("\nResume length statistics:")
print(df_clean["text_length"].describe())

# Show a few resumes
display(df_clean[["content", "text_length"]].head())

Missing resumes: 0
Empty resumes: 0

Resume length statistics:
count      536.000000
mean      3878.091418
std       3552.602342
min        346.000000
25%       1389.750000
50%       3082.500000
75%       5217.000000
max      32801.000000
Name: text_length, dtype: float64


,content,text_length
0,Afreen Jamadar\nActive member of IIIT Committe...,1240
1,Alok Khandai\nOperational Analyst (SQL DBA) En...,8384
2,Anvitha Rao\nAutomation developer\n\n- Email m...,2931
3,arjun ks\nSenior Program coordinator - oracle ...,5796
4,"Arun Elumalai\nQA Tester\n\nChennai, Tamil Nad...",2083


In [19]:
!pip install -q sentence-transformers faiss-cpu


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!pip install -q langchain-text-splitters


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " ", ""]
)

d:\Elevvo NLP Internship\tasks\6- RAG-Powered Talent Search Engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
documents = []

for candidate_id, resume in enumerate(df_clean["content"]):

    chunks = splitter.split_text(resume)

    for chunk_id, chunk in enumerate(chunks):

        documents.append({
            "candidate_id": candidate_id,
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Total chunks:", len(documents))

Total chunks: 1531


In [11]:
for doc in documents[:5]:

    print("=" * 80)
    print("Candidate:", doc["candidate_id"])
    print("Chunk:", doc["chunk_id"])
    print(doc["text"][:500])

Candidate: 0
Chunk: 0
Afreen Jamadar
Active member of IIIT Committee in Third year

Sangli, Maharashtra - Email me on Indeed: indeed.com/r/Afreen-Jamadar/8baf379b705e37c6

I wish to use my knowledge, skills and conceptual understanding to create excellent team
environments and work consistently achieving organization objectives believes in taking initiative
and work to excellence in my work.

WORK EXPERIENCE

Active member of IIIT Committee in Third year

Cisco Networking -  Kanpur, Uttar Pradesh

organized by Techkr
Candidate: 1
Chunk: 0
Alok Khandai
Operational Analyst (SQL DBA) Engineer - UNISYS

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Alok-Khandai/5be849e443b8f467

❖ Having 3.5 Years of IT experience in SQL Database Administration, System Analysis, Design,
Development & Support of MS SQL Servers in Production, Development environments &
Replication and Cluster Server Environments.
❖ Working Experience with relational database such as SQL.
❖ Experience in Installatio

In [8]:
import pandas as pd

chunks_df = pd.DataFrame(documents)

print("Total chunks:", len(chunks_df))
print("Unique candidates:", chunks_df["candidate_id"].nunique())

print("\nChunks per candidate:")
print(chunks_df.groupby("candidate_id").size().describe())

print("\nChunk length statistics:")
print(chunks_df["text"].str.len().describe())

Total chunks: 1531
Unique candidates: 536

Chunks per candidate:
count    536.000000
mean       2.856343
std        2.472253
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       22.000000
dtype: float64

Chunk length statistics:
count    1531.000000
mean     1443.383410
std       522.106037
min        38.000000
25%      1050.000000
50%      1607.000000
75%      1912.000000
max      2000.000000
Name: text, dtype: float64


In [9]:
import pickle

with open("resume_chunks.pkl", "wb") as f:
    pickle.dump(documents, f)

print("Saved:", len(documents), "chunks")

Saved: 1531 chunks


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1177.66it/s]


In [11]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

Device: cpu


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1421.05it/s]


In [12]:
texts = [doc["text"] for doc in documents]

print("Number of texts:", len(texts))
print("First text:")
print(texts[0][:500])

Number of texts: 1531
First text:
Afreen Jamadar
Active member of IIIT Committee in Third year

Sangli, Maharashtra - Email me on Indeed: indeed.com/r/Afreen-Jamadar/8baf379b705e37c6

I wish to use my knowledge, skills and conceptual understanding to create excellent team
environments and work consistently achieving organization objectives believes in taking initiative
and work to excellence in my work.

WORK EXPERIENCE

Active member of IIIT Committee in Third year

Cisco Networking -  Kanpur, Uttar Pradesh

organized by Techkr


In [13]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches: 100%|██████████| 48/48 [01:47<00:00,  2.25s/it]


In [15]:
print("Embedding shape:", embeddings.shape)

Embedding shape: (1531, 384)


In [16]:
import numpy as np

np.save("resume_embeddings.npy", embeddings)

print("Embeddings saved!")

Embeddings saved!


In [17]:
import faiss
import numpy as np

# Convert embeddings to float32
embeddings = embeddings.astype("float32")

# Create FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

# Add embeddings
index.add(embeddings)

print("Number of vectors in FAISS:", index.ntotal)

Number of vectors in FAISS: 1531


In [18]:
faiss.write_index(index, "resume_index.faiss")

print("FAISS index saved!")

FAISS index saved!


In [19]:
from collections import defaultdict

candidate_chunks = defaultdict(list)

for doc in documents:
    candidate_chunks[doc["candidate_id"]].append(doc)

print("Number of candidates:", len(candidate_chunks))

Number of candidates: 536


In [22]:
def retrieve_candidates(query, retrieval_k=15, top_n=3):

    # 1. Embed the query
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # 2. Retrieve matching chunks
    scores, indices = index.search(
        query_embedding,
        retrieval_k
    )

    # 3. Find unique candidates
    candidate_scores = {}

    for score, idx in zip(scores[0], indices[0]):

        doc = documents[idx]
        candidate_id = doc["candidate_id"]

        # Keep the highest score for each candidate
        if (
            candidate_id not in candidate_scores
            or score > candidate_scores[candidate_id]
        ):
            candidate_scores[candidate_id] = float(score)

    # 4. Rank candidates
    ranked_candidates = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # 5. Take top candidates
    top_candidates = ranked_candidates[:top_n]

    # 6. Get ALL chunks for those candidates
    results = []

    for candidate_id, best_score in top_candidates:

        results.append({
            "candidate_id": candidate_id,
            "best_score": best_score,
            "chunks": candidate_chunks[candidate_id]
        })

    return results

In [23]:
results = retrieve_candidates(
    "Find a candidate with SQL and database administration experience.",
    retrieval_k=15,
    top_n=3
)

for rank, candidate in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print("Rank:", rank)
    print("Candidate ID:", candidate["candidate_id"])
    print("Best Score:", candidate["best_score"])
    print("Total resume chunks:", len(candidate["chunks"]))

    for chunk in candidate["chunks"]:
        print("\n--- Chunk", chunk["chunk_id"], "---")
        print(chunk["text"][:300])


Rank: 1
Candidate ID: 1
Best Score: 0.6081113815307617
Total resume chunks: 5

--- Chunk 0 ---
Alok Khandai
Operational Analyst (SQL DBA) Engineer - UNISYS

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Alok-Khandai/5be849e443b8f467

❖ Having 3.5 Years of IT experience in SQL Database Administration, System Analysis, Design,
Development & Support of MS SQL Servers in Production, Dev

--- Chunk 1 ---
Willing to relocate to: Bengaluru, Karnataka

WORK EXPERIENCE

Operational Analyst (SQL DBA) Engineer

UNISYS -  Bengaluru, Karnataka -

July 2016 to Present

❖ Having 3.5 Years of IT experience in SQL Database Administration, System Analysis, Design,
Development & Support of MS SQL Servers in Produ

--- Chunk 2 ---
Roles and Responsibilities:
• Responsible for Database support, troubleshooting, planning and migration. Resource planning
and coordination for application migrations with project managers, application and web app
teams. Project involved guidance and adherence to stan

In [24]:
!pip install -q -U google-genai


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import pickle
import numpy as np
import faiss

# Load chunks
with open("resume_chunks.pkl", "rb") as f:
    documents = pickle.load(f)

# Load embeddings
embeddings = np.load("resume_embeddings.npy")

# Load FAISS index
index = faiss.read_index("resume_index.faiss")

print("Documents:", len(documents))
print("Embeddings:", embeddings.shape)
print("FAISS vectors:", index.ntotal)

Documents: 1531
Embeddings: (1531, 384)
FAISS vectors: 1531


In [6]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Say hello in two sentence."
)

print(response.text)

Hello! I hope you are having a wonderful day.


In [22]:
# Create a lookup of all chunks belonging to each candidate

candidate_chunks = {}

for doc in documents:
    candidate_id = doc["candidate_id"]

    if candidate_id not in candidate_chunks:
        candidate_chunks[candidate_id] = []

    candidate_chunks[candidate_id].append(doc["text"])

print("Candidates:", len(candidate_chunks))
print("Candidate 1 chunks:", len(candidate_chunks[1]))

Candidates: 536
Candidate 1 chunks: 5


In [23]:
def get_candidate_resume(candidate_id):
    
    if candidate_id not in candidate_chunks:
        return f"Candidate {candidate_id} does not exist."

    resume = "\n\n".join(candidate_chunks[candidate_id])

    return resume

In [24]:
resume = get_candidate_resume(1)

print(resume[:2000])

Alok Khandai
Operational Analyst (SQL DBA) Engineer - UNISYS

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Alok-Khandai/5be849e443b8f467

❖ Having 3.5 Years of IT experience in SQL Database Administration, System Analysis, Design,
Development & Support of MS SQL Servers in Production, Development environments &
Replication and Cluster Server Environments.
❖ Working Experience with relational database such as SQL.
❖ Experience in Installation, Configuration, Maintenance and Administration of SQL Server.
❖ Experience in upgrading SQL Server.
❖ Good experience with implementing DR solution, High Availability of database servers using
Database mirroring and replications and Log Shipping.
❖ Experience in implementing SQL Server security and Object permissions like maintaining
Database authentication modes, creation of users, configuring permissions and assigning roles
to users.
❖ Experience in creating Jobs, Alerts, SQL Mail Agent
❖ Experience in performing integrity checks. Meth

In [25]:
def search_candidates(query, top_n=5, chunk_k=15):
    
    # Embed the query
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # Search FAISS
    scores, indices = index.search(
        query_embedding.astype("float32"),
        chunk_k
    )

    candidate_scores = {}

    for score, idx in zip(scores[0], indices[0]):
        
        doc = documents[idx]
        candidate_id = doc["candidate_id"]

        # Keep the best matching chunk score
        if (
            candidate_id not in candidate_scores
            or score > candidate_scores[candidate_id]
        ):
            candidate_scores[candidate_id] = float(score)

    # Sort candidates by their best chunk score
    ranked_candidates = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Return top N candidates
    results = []

    for candidate_id, score in ranked_candidates[:top_n]:

        results.append({
            "candidate_id": candidate_id,
            "score": score,
            "resume": get_candidate_resume(candidate_id)
        })

    return results

In [26]:
results = search_candidates(
    "people with deep learning experience",
    top_n=5
)

for i, candidate in enumerate(results, 1):

    print("=" * 80)
    print("Candidate:", i)
    print("Candidate ID:", candidate["candidate_id"])
    print("Score:", candidate["score"])
    print("=" * 80)

    print(candidate["resume"][:500])

Candidate: 1
Candidate ID: 514
Score: 0.3948025107383728
Deepti Bansal
Senior Software Engineer mentored freshers

- Email me on Indeed: indeed.com/r/Deepti-Bansal/67ff5e5e71ff17ca

WORK EXPERIENCE

Senior Software Engineer

Aricent India -  Gurgaon, Haryana -

June 2012 to October 2015

US
Duration Oct2012 - Oct2015
Team Size 110
Cisco Unified Communications Solutions unify voice, video, data, and mobile applications on fixed
and mobile networks, enabling easy collaboration every time from any workspace.

Development Environment
• Language - C++
• Pl
Candidate: 2
Candidate ID: 41
Score: 0.33573222160339355
Mohini Gupta
Server Support Engineer

Gurgaon, Haryana - Email me on Indeed: indeed.com/r/Mohini-Gupta/08b5b8e1acd8cf07

Willing to relocate: Anywhere

WORK EXPERIENCE

Server Support Engineer

Microsoft -

July 2015 to November 2017

Key Responsibilities:
● Worked as Technical Support Engineer for Microsoft Enterprise Platforms Support.

● Worked in U.S., U.K., India & APAC time zo

In [27]:
conversation_state = {
    "current_candidates": {},
    "history": []
}

In [28]:
def save_search_results(results):

    conversation_state["current_candidates"] = {}

    for rank, candidate in enumerate(results, 1):

        conversation_state["current_candidates"][rank] = candidate["candidate_id"]

In [29]:
def get_candidate_by_rank(rank):

    candidates = conversation_state["current_candidates"]

    if rank not in candidates:
        return None

    candidate_id = candidates[rank]

    return {
        "candidate_id": candidate_id,
        "resume": get_candidate_resume(candidate_id)
    }

In [30]:
save_search_results(results)

print(conversation_state["current_candidates"])

{1: 514, 2: 41, 3: 401, 4: 117, 5: 138}


In [31]:
candidate = get_candidate_by_rank(1)

print("Candidate ID:", candidate["candidate_id"])
print()
print(candidate["resume"][:1000])

Candidate ID: 514

Deepti Bansal
Senior Software Engineer mentored freshers

- Email me on Indeed: indeed.com/r/Deepti-Bansal/67ff5e5e71ff17ca

WORK EXPERIENCE

Senior Software Engineer

Aricent India -  Gurgaon, Haryana -

June 2012 to October 2015

US
Duration Oct2012 - Oct2015
Team Size 110
Cisco Unified Communications Solutions unify voice, video, data, and mobile applications on fixed
and mobile networks, enabling easy collaboration every time from any workspace.

Development Environment
• Language - C++
• Platform - Linux
• Version Control - Clearcase, GIT
• Tools - Source Insight, Eclipse, Wireshark, CDETS, Rally
• Automation tools - GTest, Gcov

Roles and Responsibilities
• As Senior Software Engineer mentored freshers to fast-track learning - Trained basic
professional as well as softer skills necessary on the job
• Proficient in Agile WoW, having development experience of about 3 years
• Development experience in C++ while working on multiple featues of Unity Connection
• Alw

In [32]:
from google.genai import types

In [ ]:
search_tool = types.FunctionDeclaration(
    name="search_candidates",
    description="""
    Search the resume database for candidates matching the recruiter's
    requirements. Use this when the recruiter asks to find, search for,
    rank, or recommend candidates.
    """,
    parameters={
        "type": "OBJECT",
        "properties": {
            "query": {
                "type": "STRING",
                "description": "The recruiter's requirements."
            },
            "top_n": {
                "type": "INTEGER",
                "description": """
                Number of candidates requested by the recruiter.
                If they specify a number, use that number.
                If they do not specify a number, use 5.
                """
            }
        },
        "required": ["query"]
    }
)


candidate_tool = types.FunctionDeclaration(
    name="get_candidate",
    description="""
    Retrieve the complete resume of a candidate from the current search
    results. Use this when the recruiter asks about a specific candidate
    or asks follow-up questions about a candidate.
    """,
    parameters={
        "type": "OBJECT",
        "properties": {
            "rank": {
                "type": "INTEGER",
                "description": """
                Candidate rank from the most recent search.
                Candidate 1 means the first candidate in the latest results.
                """
            }
        },
        "required": ["rank"]
    }
)


compare_tool = types.FunctionDeclaration(
    name="compare_candidates",
    description="""
    Compare two candidates from the current search results.
    Use this when the recruiter asks to compare candidates.
    """,
    parameters={
        "type": "OBJECT",
        "properties": {
            "candidate_a": {
                "type": "INTEGER",
                "description": "First candidate rank."
            },
            "candidate_b": {
                "type": "INTEGER",
                "description": "Second candidate rank."
            }
        },
        "required": ["candidate_a", "candidate_b"]
    }
)

In [34]:
agent_tools = types.Tool(
    function_declarations=[
        search_tool,
        candidate_tool,
        compare_tool
    ]
)

In [35]:
agent_config = types.GenerateContentConfig(
    temperature=0.2,
    tools=[agent_tools],

    system_instruction="""
You are an AI recruitment assistant connected to a resume database.

Your job is to help recruiters find and evaluate candidates.

IMPORTANT RULES:

1. When the recruiter asks to FIND or SEARCH for candidates,
   use the search_candidates tool.

2. The recruiter may request any number of candidates.
   Examples:
   - "Find 3 candidates"
   - "Find 5 candidates"
   - "Find 10 candidates"
   - "Find 20 candidates"

   Always respect the requested number.

3. If the recruiter does not specify a number,
   search for 5 candidates by default.

4. Candidate numbers refer to the rankings from the MOST RECENT search.

5. When the recruiter asks:
   "Tell me more about candidate 1"
   use get_candidate with rank=1.

6. When the recruiter says:
   "Does he have leadership experience?"
   "What are her skills?"
   "Where does he work?"
   resolve the pronoun using the conversation context
   and retrieve the appropriate candidate.

7. When the recruiter asks to compare candidates,
   use compare_candidates.

8. Base your answers ONLY on information present in the resumes.

9. Never invent skills, experience, education, leadership,
   certifications, or other qualifications.

10. If something is not mentioned in the resume,
    say that it is not explicitly mentioned.

11. Similarity scores are retrieval signals, NOT proof that
    a candidate has a particular skill.

12. Give useful recruiter-friendly explanations rather than
    simply returning numerical similarity scores.
"""
)

In [36]:
def execute_tool(tool_name, args):

    if tool_name == "search_candidates":

        query = args["query"]
        top_n = args.get("top_n", 5)

        results = search_candidates(
            query=query,
            top_n=top_n,
            chunk_k=30
        )

        # Save candidates for conversation context
        save_search_results(results)

        return results


    elif tool_name == "get_candidate":

        rank = args["rank"]

        candidate = get_candidate_by_rank(rank)

        if candidate is None:
            return {
                "error": f"Candidate {rank} does not exist in the current search."
            }

        return candidate


    elif tool_name == "compare_candidates":

        rank_a = args["candidate_a"]
        rank_b = args["candidate_b"]

        candidate_a = get_candidate_by_rank(rank_a)
        candidate_b = get_candidate_by_rank(rank_b)

        if candidate_a is None or candidate_b is None:
            return {
                "error": "One or both candidates do not exist."
            }

        return {
            "candidate_a": candidate_a,
            "candidate_b": candidate_b
        }

    return {
        "error": f"Unknown tool: {tool_name}"
    }

In [40]:
conversation_history = []


def recruiter_agent(user_message):

    # Add user's message
    conversation_history.append({
        "role": "user",
        "parts": [
            {"text": user_message}
        ]
    })

    while True:

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=conversation_history,
            config=agent_config
        )

        # --------------------------------------------------
        # Check whether Gemini wants to use a tool
        # --------------------------------------------------

        function_calls = []

        for part in response.candidates[0].content.parts:

            if part.function_call:
                function_calls.append(part.function_call)

        # --------------------------------------------------
        # No tool call = final answer
        # --------------------------------------------------

        if not function_calls:

            conversation_history.append({
                "role": "model",
                "parts": response.candidates[0].content.parts
            })

            return response.text

        # --------------------------------------------------
        # Gemini requested one or more tools
        # --------------------------------------------------

        conversation_history.append({
            "role": "model",
            "parts": response.candidates[0].content.parts
        })

        tool_parts = []

        for function_call in function_calls:

            tool_name = function_call.name
            args = dict(function_call.args)

            print(
                f"\nTool: {tool_name}"
            )

            print(
                f"Arguments: {args}"
            )

            # ----------------------------------------------
            # Execute the tool
            # ----------------------------------------------

            result = execute_tool(
                tool_name,
                args
            )

            print(
                f"Tool result type: {type(result)}"
            )

            # ----------------------------------------------
            # Give result back to Gemini
            # ----------------------------------------------

            tool_parts.append({
                "function_response": {
                    "name": tool_name,
                    "response": {
                        "result": result
                    }
                }
            })

        # Add tool results to conversation
        conversation_history.append({
            "role": "user",
            "parts": tool_parts
        })

In [41]:
response = recruiter_agent(
    "Find me 5 people with SQL and database administration experience."
)

print(response)


Tool: search_candidates
Arguments: {'top_n': 5, 'query': 'SQL and database administration experience'}
Tool result type: <class 'list'>
Here are 5 candidates with SQL and database administration experience from the database:

---

### **1. Alok Khandai** (Rank 1)
* **Current/Recent Role:** Operational Analyst (SQL DBA) Engineer at UNISYS
* **Key Experience & Skills:**
  * 3.5 years of IT experience in **SQL Database Administration**, MS SQL Server installation, configuration, maintenance, and upgrading.
  * Experience with High Availability/Disaster Recovery (DR) solutions including Database Mirroring, Replication, and Log Shipping.
  * Proficient in SQL Server security, performance tuning (SQL Profiler, Performance Monitor), query optimization, and DBCC utilities.
  * Experience supporting large-scale enterprise environments (e.g., Microsoft Corporation, Costco).

---

### **2. Venkateswara D** (Rank 2)
* **Current/Recent Role:** Specialist / Production Support Analyst at Infosys Lim

In [82]:
response = recruiter_agent(
    "Tell me more about candidate 1."
)

print(response)

Tool: get_candidate Arguments: {'rank': 1}
Tool result type: <class 'dict'>
Here is a detailed breakdown of **Candidate 1: Alok Khandai**:

---

### **Overview**
* **Current Title:** Operational Analyst (SQL DBA) Engineer at UNISYS
* **Location:** Bengaluru, Karnataka (Willing to relocate)
* **Total Experience:** 3.5+ years of IT experience in SQL Database Administration, system analysis, design, development, and support of MS SQL Servers across production, development, replication, and clustered environments.

---

### **Education**
* **B.Tech in Computer Science and Engineering** – Indira Gandhi Institute of Technology (2012)

---

### **Work Experience & Key Projects**

1. **UNISYS** | *Operational Analyst (SQL DBA) Engineer* (July 2016 – Present)
   * Responsible for MS SQL Server administration, installation, configuration, maintenance, and upgrades.
   * Implemented High Availability (HA) and Disaster Recovery (DR) solutions using Database Mirroring, Transactional Replication, an

In [83]:
response = recruiter_agent(
    "Does he have leadership experience?"
)

print(response)

Based on Alok Khandai's resume, **leadership experience is not explicitly mentioned**.

His resume details technical roles as a DBA Support Analyst and Operational Analyst (SQL DBA) Engineer, where he worked within teams (including a project team of 22) and coordinated with project managers and application teams. However, there is no explicit record of people management, team leadership, or supervisory roles.


In [84]:
response = recruiter_agent(
    "Compare him with candidate 3."
)

print(response)

Tool: compare_candidates Arguments: {'candidate_a': 1, 'candidate_b': 3}
Tool result type: <class 'dict'>
Here is a side-by-side comparison of **Candidate 1 (Alok Khandai)** and **Candidate 3 (Nidhi Pandit)** based on their resume details:

---

### **1. Core Role & Focus**
* **Candidate 1 (Alok Khandai):** Dedicated **SQL Database Administrator (DBA)**. His career is focused entirely on managing, maintaining, and supporting production MS SQL Server environments.
* **Candidate 3 (Nidhi Pandit):** **Software Test Engineer (QA)**. Her core focus is on manual and automated software testing (REST APIs, Telecom/Banking systems), using SQL primarily for test data preparation and verification.

---

### **2. Database Administration (DBA) Experience**
* **Candidate 1 (Alok Khandai):**
  * **Strong DBA Experience:** Has 3.5+ years of hands-on DBA experience (installations, upgrades, clustering, log shipping, database mirroring, replication).
  * **Performance & Maintenance:** Skilled in perform

In [ ]:
response = recruiter_agent(
    "Find me 3 people with deep learning experience."
)

print(response)

In [ ]:
response = recruiter_agent(
    "Find me strongest 1 with deep learning experience."
)

print(response)